In [1]:
from pyspark.sql import SparkSession
import getpass

username = getpass.getuser()

In [2]:
spark = SparkSession.builder \
.config("spark.port.ui", 0) \
.config("spark.sql.warehouse.dir", f"/user/{username}/warehouse") \
.enableHiveSupport() \
.master("yarn") \
.getOrCreate()

In [3]:
from pyspark.sql.functions import expr, rand, round

# 1. Generate 150 distinct stores natively on the executors
stores_df = spark.range(1, 151).withColumn("store_id", expr("concat('Store_', id)")).drop("id")

# 2. Generate 730 continuous days (2 years) using Spark SQL sequence
dates_df = spark.sql("SELECT explode(sequence(to_date('2022-01-01'), to_date('2023-12-31'), interval 1 day)) as sales_date")

# 3. Cross join to ensure every store has exactly one record per day, then add random revenue
df_large = stores_df.crossJoin(dates_df) \
    .withColumn("daily_revenue", round((rand() * 4900) + 100, 2))

print(f"Total Rows: {df_large.count()}")
df_large.show(5)

Total Rows: 109500


In [6]:
from pyspark.sql import Window

window_spec = Window.partitionBy("store_id").orderBy("sales_date").rowsBetween(-6, Window.currentRow)

In [11]:
from pyspark.sql import functions as F

avg_sales_df = df_large.withColumn("7_day_rolling_avg", F.avg("daily_revenue").over(window_spec))

In [12]:
avg_sales_df.show()

+--------+----------+-------------+------------------+
|store_id|sales_date|daily_revenue| 7_day_rolling_avg|
+--------+----------+-------------+------------------+
| Store_2|2022-01-01|      4722.74|           4722.74|
| Store_2|2022-01-02|      2559.34|           3641.04|
| Store_2|2022-01-03|      1251.77| 2844.616666666667|
| Store_2|2022-01-04|       3610.9|         3036.1875|
| Store_2|2022-01-05|      3219.92|          3072.934|
| Store_2|2022-01-06|       659.21| 2670.646666666667|
| Store_2|2022-01-07|       479.68| 2357.651428571429|
| Store_2|2022-01-08|       896.59|1811.0585714285714|
| Store_2|2022-01-09|      1064.59|1597.5228571428572|
| Store_2|2022-01-10|      3773.92|           1957.83|
| Store_2|2022-01-11|      2505.11|1799.8600000000001|
| Store_2|2022-01-12|      4301.57|1954.3814285714286|
| Store_2|2022-01-13|       345.27| 1909.532857142857|
| Store_2|2022-01-14|      1115.51|2000.3657142857144|
| Store_2|2022-01-15|      4277.21|2483.3114285714287|
| Store_2|